# Lab 4 — Trực quan quy trình Wavelet Hash
Chạy **Run All** từ trên xuống. Notebook dùng lại `wavelet_lab.py`, không triển khai một thuật toán khác.

**Ảnh → EXIF/grayscale/resize → phân rã Wavelet → LL 8×8 → lượng tử hóa median → 64 bit → Hamming → đánh giá / tìm kiếm.**

Ngưỡng được chọn trên validation; chỉ số cuối cùng được tính trên test. Dữ liệu demo minh họa ảnh gần trùng, chưa đại diện cho nhận diện cùng đối tượng ở nhiều góc chụp.

In [ ]:
# Chạy cell này nếu kernel chưa có thư viện.
%pip install numpy Pillow PyWavelets scikit-learn matplotlib

In [ ]:
from pathlib import Path
import sys, csv, json
import numpy as np
import pywt
from PIL import Image, ImageOps
from IPython.display import display, Markdown

# Hỗ trợ mở notebook từ Lab_4 hoặc thư mục gốc repository.
LAB = Path.cwd().resolve()
if not (LAB / 'wavelet_lab.py').is_file():
    LAB = LAB / 'Lab_4'
if not (LAB / 'wavelet_lab.py').is_file():
    raise FileNotFoundError('Hãy mở notebook từ Lab_4 hoặc thư mục gốc repository.')
sys.path.insert(0, str(LAB))
from wavelet_lab import prepare_demo, read_manifest, wavelet_hash, hamming_distance, evaluate, search
%matplotlib inline
import matplotlib.pyplot as plt
DATA = LAB / 'data' / 'demo'
OUT = LAB / 'outputs' / 'notebook'
OUT.mkdir(parents=True, exist_ok=True)
print('Project:', LAB)

## 1. Chuẩn bị và quan sát dữ liệu
Hai ảnh mẫu đi kèm scikit-learn được chia thành 12 crop nguồn. Mỗi crop có 5 phiên bản: gốc, sáng hơn, nhiễu, giảm kích thước, JPEG. Seed = 42. Các biến thể cùng crop luôn ở cùng split.

Hai ảnh cùng `group` là cặp dương. Khác `group` là cặp âm. Các crop vẫn đến từ hai ảnh chụp nên bộ demo còn hạn chế về độ đa dạng.

In [ ]:
prepare_demo(DATA)
rows = read_manifest(DATA / 'manifest.csv')
for split in ('validation', 'test'):
    subset = [r for r in rows if r['split'] == split]
    print(f'{split}: {len(subset)} ảnh, {len(set(r["group"] for r in subset))} nhóm')
group_rows = [r for r in rows if r['group'] == rows[0]['group']]
fig, axes = plt.subplots(1, len(group_rows), figsize=(16, 4))
for ax, row in zip(axes, group_rows):
    with Image.open(row['file']) as image:
        ax.imshow(image)
    ax.set_title(Path(row['path']).stem.split('_')[-1]); ax.axis('off')
plt.suptitle('Các biến thể của cùng một ảnh nguồn'); plt.tight_layout(); plt.show()

## 2. Tiền xử lý và các thành phần Wavelet
Ảnh được sửa hướng EXIF, chuyển grayscale, resize 128×128 và chuẩn hóa [0,1]. Hình dưới minh họa phân rã **một mức**: LL chứa cấu trúc tần số thấp, các thành phần chi tiết biểu diễn biến thiên theo hướng. Khi tạo hash, chương trình phân rã **bốn mức** để nhận LL 8×8.

In [ ]:
WAVELET = 'haar'  # Có thể đổi thành 'db2' hoặc 'db4', rồi chạy lại các cell sau.
query = DATA / 'photo0_crop0_original.jpg'
with Image.open(query) as source:
    rgb = ImageOps.exif_transpose(source).convert('RGB')
    gray = rgb.convert('L').resize((128, 128), Image.Resampling.LANCZOS)
pixels = np.asarray(gray, dtype=float) / 255.0
ll1, (horizontal, vertical, diagonal) = pywt.dwt2(pixels, WAVELET, mode='periodization')
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
panels = [(rgb, 'Ảnh gốc'), (pixels, 'Grayscale 128×128'), (ll1, 'LL — xấp xỉ'),
          (horizontal, 'Chi tiết ngang'), (vertical, 'Chi tiết dọc'), (diagonal, 'Chi tiết chéo')]
for ax, (values, title) in zip(axes.ravel(), panels):
    ax.imshow(values, cmap='gray'); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()
print('Các panel được tự chuẩn hóa mức hiển thị để dễ nhìn chi tiết.')

## 3. Từ hệ số LL sang mã băm
Lấy LL sau bốn mức. Bit bằng 1 nếu hệ số lớn hơn median, ngược lại bằng 0. Cùng kích thước 8×8 nên các hash có thể so sánh trực tiếp.

In [ ]:
ll = pywt.wavedec2(pixels, WAVELET, mode='periodization', level=4)[0]
median = float(np.median(ll))
bits = wavelet_hash(query, WAVELET)
assert np.array_equal(bits, (ll > median).ravel())
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
im = axes[0].imshow(ll, cmap='viridis'); axes[0].set_title('LL sau 4 mức (8×8)')
fig.colorbar(im, ax=axes[0], fraction=0.046)
axes[1].hist(ll.ravel(), bins=16); axes[1].axvline(median, color='red', label=f'median={median:.3f}')
axes[1].set_title('Phân bố hệ số'); axes[1].legend()
axes[2].imshow(bits.reshape(8, 8), cmap='gray', vmin=0, vmax=1); axes[2].set_title('Hash: đen=0, trắng=1')
plt.tight_layout(); plt.show()
print('Hash 64 bit:', ''.join(bits.astype(int).astype(str)))

## 4. So sánh Hamming
Khoảng cách = số bit khác nhau / 64. Giá trị càng thấp thì hash càng giống nhau. Hình XOR tô trắng những bit khác. Hash chỉ phản ánh cấu trúc ảnh, không bảo đảm tương đồng ngữ nghĩa.

In [ ]:
candidates = [DATA / 'photo0_crop0_noise.jpg', DATA / 'photo1_crop5_original.jpg']
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for axes_row, candidate in zip(axes, candidates):
    other = wavelet_hash(candidate, WAVELET)
    distance = hamming_distance(bits, other)
    with Image.open(candidate) as image:
        axes_row[0].imshow(image)
    axes_row[0].set_title(candidate.stem)
    axes_row[1].imshow(other.reshape(8, 8), cmap='gray', vmin=0, vmax=1)
    axes_row[1].set_title('Hash ứng viên')
    axes_row[2].imshow((bits != other).reshape(8, 8), cmap='gray', vmin=0, vmax=1)
    axes_row[2].set_title(f'XOR: {np.count_nonzero(bits != other)}/64, d={distance:.3f}')
    for ax in axes_row: ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Đánh giá và so sánh ba họ Wavelet
Mỗi phương pháp tự chọn ngưỡng trên validation bằng balanced accuracy; sau đó cố định ngưỡng để đánh giá test. ROC dùng điểm liên tục `1 − distance`.

- Accuracy = (TP+TN)/tổng số cặp.
- Sensitivity (recall) = TP/(TP+FN).
- Specificity = TN/(TN+FP), khác precision = TP/(TP+FP).

Các cặp âm nhiều hơn cặp dương; cần đọc đồng thời recall, specificity và AUC. Không chọn lại ngưỡng dựa trên test.

In [ ]:
evaluate(DATA / 'manifest.csv', OUT)
metrics = json.loads((OUT / 'metrics.json').read_text(encoding='utf-8'))
columns = ['wavelet', 'threshold', 'accuracy', 'sensitivity', 'specificity', 'precision', 'roc_auc']
table = '| ' + ' | '.join(columns) + ' |\n|' + '|'.join(['---'] * len(columns)) + '|\n'
for row in metrics:
    table += '| ' + ' | '.join(f'{row[c]:.4f}' if isinstance(row[c], float) else str(row[c]) for c in columns) + ' |\n'
display(Markdown(table))
with Image.open(OUT / 'roc.png') as chart:
    display(chart.copy())
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, name in zip(axes, ('haar', 'db2', 'db4')):
    with Image.open(OUT / f'confusion_{name}.png') as chart:
        ax.imshow(chart)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Quan sát vùng chồng lấn khoảng cách giữa cặp dương và cặp âm trên test.
with (OUT / f'pairs_{WAVELET}.csv').open(encoding='utf-8', newline='') as stream:
    pairs = list(csv.DictReader(stream))
threshold = next(r['threshold'] for r in metrics if r['wavelet'] == WAVELET)
fig, ax = plt.subplots(figsize=(9, 4))
for label, title in [('1', 'Tương tự'), ('0', 'Không tương tự')]:
    values = [float(p['distance']) for p in pairs if p['label'] == label]
    ax.hist(values, bins=np.linspace(0, 1, 33), alpha=0.55, density=True, label=title)
ax.axvline(threshold, color='black', linestyle='--', label=f'Ngưỡng validation={threshold:.3f}')
ax.set(xlabel='Hamming chuẩn hóa', ylabel='Mật độ', title=f'Khoảng cách trên test — {WAVELET}')
ax.legend(); plt.show()
errors = [p for p in pairs if (float(p['distance']) <= threshold) != bool(int(p['label']))]
print(f'Số cặp phân loại sai: {len(errors)}/{len(pairs)}')
for p in errors[:5]: print(p)

## 6. Tìm ảnh tương tự top-k
Tạo hash truy vấn, so sánh với từng ảnh và sắp xếp Hamming tăng dần. Chính tệp truy vấn được loại khỏi ứng viên. Điểm tương tự `1 − distance` không phải xác suất.

In [ ]:
search(query, DATA, WAVELET, 5, OUT / 'search.csv')
with (OUT / 'search.csv').open(encoding='utf-8', newline='') as stream:
    matches = list(csv.DictReader(stream))
fig, axes = plt.subplots(1, len(matches) + 1, figsize=(18, 4))
with Image.open(query) as image: axes[0].imshow(image)
axes[0].set_title('Truy vấn'); axes[0].axis('off')
for rank, (ax, match) in enumerate(zip(axes[1:], matches), start=1):
    with Image.open(match['path']) as image: ax.imshow(image)
    ax.set_title(f'Top {rank}\nd={float(match["distance"]):.3f}')
    ax.axis('off')
plt.tight_layout(); plt.show()
print('Kết quả lưu tại:', OUT)

## 7. Câu hỏi phân tích và mở rộng
1. Haar, db2, db4 khác nhau thế nào về recall, specificity và AUC? Dùng bảng thực tế để trả lời.
2. Các cặp sai có bố cục hoặc độ sáng giống nhau không? Mở ảnh theo đường dẫn trong `errors`.
3. Thử ảnh xoay hoặc crop: hash có còn ổn định? Những biến đổi này chưa được bộ demo đánh giá.
4. Thay dữ liệu demo bằng manifest ảnh thật theo README; chia đối tượng/ảnh nguồn độc lập giữa validation và test.
5. Không suy ra độ chính xác ngoài thực tế chỉ từ hai ảnh mẫu. Các cặp chia sẻ ảnh cũng không độc lập thống kê.

Notebook được lưu chưa có output; các bảng và hình chỉ được sinh khi chạy, không có số liệu kết quả giả định.